# Deepfake Voice Detection Using Mel Spectrograms and Convolutional Neural Networks (CNN)

Deepfake audio generated using modern AI models has become increasingly realistic, making it difficult to distinguish synthetic speech from genuine human speech. Such technologies can be misused for misinformation, fraud, identity impersonation, and social engineering attacks.

The objective of this project is to develop a Deep Learning-based system capable of detecting AI-generated speech from raw audio recordings. The proposed pipeline converts audio signals into Mel Spectrogram representations and uses a Convolutional Neural Network (CNN) to classify audio samples as REAL or FAKE.

The project follows a complete end-to-end workflow including audio preprocessing, feature extraction, model training, evaluation, and deployment through a Streamlit web application.


In [ ]:
import pandas as pd
import librosa
import scipy
import IPython.display as ipd
import numpy as np
import matplotlib.pyplot as plt
import os

# Dataset Overview

The dataset consists of real and AI-generated speech recordings stored in WAV format. Real audio files contain genuine human speech, while fake audio files are generated through voice conversion techniques that imitate the voices of target speakers.

Each audio recording is several minutes long. To increase the number of training samples and allow the model to learn local speech patterns, long recordings are divided into fixed-length one-second chunks before feature extraction.


# Loading and Exploring Audio Data

Before training any Deep Learning model, it is important to understand the structure of the dataset. In this section, audio files are loaded and basic statistics such as duration, sample rate, and class distribution are analyzed.

Understanding the dataset characteristics helps identify issues such as class imbalance, missing data, and inconsistencies between recordings.


In [ ]:
real_folder = "data/KAGGLE/AUDIO/REAL"

real_files = [
    os.path.join(real_folder, file)
    for file in os.listdir(real_folder)
    if file.endswith(".wav")
]

len(real_files)

In [ ]:
fake_folder = "data/KAGGLE/AUDIO/FAKE"

fake_files = [
    os.path.join(fake_folder, file)
    for file in os.listdir(fake_folder)
    if file.endswith(".wav")
]

len(fake_files)

In [ ]:
X_audio = []
y = []

# Splitting Audio into One-Second Chunks

Deep Learning models require a large number of training samples. Instead of treating each audio recording as a single sample, every audio file is divided into one-second segments.

This approach significantly increases the dataset size and allows the model to learn local acoustic patterns that may reveal deepfake artifacts. Each chunk becomes an independent training sample for the classification model.


In [ ]:
for file in real_files:

    audio, sr = librosa.load(
        file, sr = 16000
    )

    for i in range(0, len(audio), sr):
        
        chunk = audio[i:i+sr]
        
        if(len(chunk)==sr):
            X_audio.append(chunk)
            y.append(0)

In [ ]:
for file in fake_files:

    audio, sr = librosa.load(
        file, sr = 16000
    )

    for i in range(0, len(audio), sr):
        
        chunk = audio[i:i+sr]
        
        if(len(chunk)==sr):
            X_audio.append(chunk)
            y.append(1)

In [ ]:
X_audio = np.array(X_audio, dtype=np.float32)
y = np.array(y)

print(X_audio.shape)
print(y.shape)

In [ ]:
unique, counts = np.unique(y, return_counts=True)

for cls, count in zip(unique, counts):
    print(cls, count)

# Handling Class Imbalance

After chunking, the number of fake samples became significantly larger than the number of real samples. Training a model on such an imbalanced dataset may cause it to become biased toward the majority class.

To ensure fair learning, the dataset is balanced through random downsampling of the majority class. This allows the model to learn meaningful patterns from both classes equally.


In [ ]:
real_idx = np.where(y==0)[0]
fake_idx = np.where(y==1)[0]

fake_idx = np.random.choice(
    fake_idx,
    size=len(real_idx),
    replace=False
)

selected_idx = np.concatenate([
    real_idx, fake_idx
])

np.random.shuffle(selected_idx)

X_audio_balanced = X_audio[selected_idx]
y_balanced = y[selected_idx]

In [ ]:
unique, counts = np.unique(y_balanced, return_counts=True)

for cls, count in zip(unique, counts):
    print(cls, count)

# Converting Audio into Mel Spectrograms

Raw audio waveforms contain thousands of amplitude values and are not directly suitable for CNN-based image processing.

A Mel Spectrogram transforms the audio signal into a two-dimensional representation where:

* The horizontal axis represents time.
* The vertical axis represents frequency.
* Pixel intensity represents signal energy.

This representation preserves important speech characteristics while making the data suitable for Convolutional Neural Networks.


# Applying Logarithmic Scaling

Human hearing perceives sound on a logarithmic scale rather than a linear scale. Therefore, Mel Spectrograms are converted into decibel (dB) units using logarithmic scaling.

This transformation improves the visibility of important speech patterns and often leads to better Deep Learning performance.


In [ ]:
X = []

for chunk in X_audio_balanced:

    mel = librosa.feature.melspectrogram(
        y = chunk,
        sr = 16000,
        n_mels=128
    )

    mel_db = librosa.power_to_db(mel, ref=np.max)
    X.append(mel_db)

In [ ]:
np.shape(X)

# Visualising the Mel spectrogram of a Real Voice

In [ ]:
mel = librosa.feature.melspectrogram(
    y = X_audio[y==0][0],
    sr = 16000,
    n_mels=128
)

mel_db = librosa.power_to_db(mel, ref=np.max)

plt.figure(figsize=(10, 4))

librosa.display.specshow(
    mel_db,
    sr = 16000,
    x_axis='time',
    y_axis='mel'
)

plt.colorbar()
plt.title("Real Audio Mel Spectrogram")
plt.show

# Visualising the Mel spectrogram of a Fake Voice

In [ ]:
mel = librosa.feature.melspectrogram(
    y = X_audio[y==1][0],
    sr = 16000,
    n_mels=128
)

mel_db = librosa.power_to_db(mel, ref=np.max)

plt.figure(figsize=(10, 4))

librosa.display.specshow(
    mel_db,
    sr = 16000,
    x_axis='time',
    y_axis='mel'
)

plt.colorbar()
plt.title("Fake audio Mel Spectrogram")
plt.show

In [ ]:
X = np.array(X, dtype=np.float32)

In [ ]:
np.shape(X), np.shape(y_balanced)

In [ ]:
X = X.reshape(7484, 128, 32, 1)
np.shape(X)

In [ ]:
np.save("X.npy", X.astype(np.float32))
np.save("y_balanced.npy", y_balanced)
np.save("X_audio_balanced.npy", X_audio.astype(np.float32))

# Data Normalization

Neural Networks train more efficiently when input values are within a consistent numerical range.

The spectrogram values are normalized to improve optimization stability, accelerate convergence, and prevent certain features from dominating the learning process.


In [ ]:
X = (X - X.min())/(X.max()-X.min())

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y_balanced, test_size=0.2, stratify=y_balanced, random_state=42)

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
from keras.models import Sequential
from keras.layers import *
from keras.callbacks import *

# CNN Model Architecture

A Convolutional Neural Network (CNN) is used to learn spatial patterns from Mel Spectrogram images.

The network consists of multiple convolutional layers for feature extraction, batch normalization layers for training stability, pooling layers for dimensionality reduction, and fully connected layers for final classification.

The output layer uses a sigmoid activation function to predict the probability that a given audio sample is fake.


In [ ]:
model = Sequential()

model.add(Conv2D(32, (3,3),activation='relu', padding='same', input_shape=(128, 32, 1)))
model.add(BatchNormalization())
model.add(MaxPooling2D(2,2))

model.add(Conv2D(64, (3,3),activation='relu', padding='same'))
model.add(BatchNormalization())
model.add(MaxPooling2D(2,2))

model.add(Conv2D(128, (3,3),activation='relu', padding='same'))
model.add(BatchNormalization())
model.add(MaxPooling2D(2,2))

model.add(Flatten())

model.add(Dense(128, activation='relu'))
model.add(Dropout(0.3))

model.add(Dense(64, activation='relu'))
model.add(Dropout(0.3))

model.add(Dense(1, activation='sigmoid'))

In [ ]:
model.summary()

# Model Training Strategy

The model is trained using binary cross-entropy loss and the Adam optimizer.

Several techniques are employed to improve training performance:

* Early Stopping to prevent overfitting.
* Reduce Learning Rate on Plateau to improve convergence.
* Model Checkpointing to save the best-performing model based on validation loss.

These techniques help achieve better generalization on unseen data.


In [ ]:
model.compile(optimizer="Adam", loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3
)

checkpoint = ModelCheckpoint(
    "models/best_model.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

In [ ]:
history = model.fit(X_train, y_train, validation_split=0.2, epochs=30, batch_size=32, callbacks=[early_stopping, reduce_lr, checkpoint])

# Evaluating Model Performance

Model performance is evaluated using multiple metrics including:

* Accuracy
* Precision
* Recall
* F1 Score
* Confusion Matrix

These metrics provide a comprehensive understanding of the model's ability to correctly identify both real and fake audio samples.


In [ ]:
print(history.history['accuracy'])

In [ ]:
print(history.history['val_accuracy'])

In [ ]:
model.evaluate(
    X_test,
    y_test
)

In [ ]:
from keras.models import load_model

best_model = load_model("models/best_model.keras")

best_model.evaluate(
    X_test,
    y_test
)

In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

In [ ]:
y_pred = np.where(model.predict(X_test)>0.5, 1, 0)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm

In [ ]:
print(classification_report(y_test, y_pred))

# Training and Validation Loss

Loss curves provide a more detailed view of model performance than accuracy alone.

While accuracy only indicates whether predictions are correct, loss also measures the confidence of those predictions. A decreasing validation loss typically indicates improved generalization, whereas a rising validation loss may signal overfitting.

The best-performing model is selected based on the lowest validation loss observed during training.


In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'validation'], loc='upper left')
plt.show()

# Training and Validation Accuracy

The accuracy curves provide insight into the learning behavior of the model throughout training.

Training accuracy measures performance on samples seen during optimization, while validation accuracy evaluates the model on previously unseen data. A small gap between the two curves generally indicates good generalization and limited overfitting.

Monitoring these trends helps determine whether the model is learning meaningful patterns or merely memorizing the training data.


In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'validation'], loc='upper left')
plt.show()

In [ ]:
model.save_weights("models/weights_best.weights.h5")

# Recreating the CNN Architecture

To ensure reproducibility and support deployment, the model architecture is recreated programmatically. This step allows the network structure to be reconstructed independently of the training process and enables loading of pre-trained weights for inference.

The architecture consists of multiple convolutional blocks for hierarchical feature extraction, followed by fully connected layers for binary classification. Batch Normalization is used to stabilize training, while Dropout layers help reduce overfitting and improve generalization performance.


In [ ]:
model_2 = Sequential()

model_2.add(Conv2D(32, (3,3),activation='relu', padding='same', input_shape=(128, 32, 1)))
model_2.add(BatchNormalization())
model_2.add(MaxPooling2D(2,2))

model_2.add(Conv2D(64, (3,3),activation='relu', padding='same'))
model_2.add(BatchNormalization())
model_2.add(MaxPooling2D(2,2))

model_2.add(Conv2D(128, (3,3),activation='relu', padding='same'))
model_2.add(BatchNormalization())
model_2.add(MaxPooling2D(2,2))

model_2.add(Conv2D(256, (3,3),activation='relu', padding='same'))
model_2.add(BatchNormalization())
model_2.add(MaxPooling2D(2,2))

model_2.add(Flatten())

model_2.add(Dense(128, activation='relu'))
model_2.add(Dropout(0.4))

model_2.add(Dense(64, activation='relu'))
model_2.add(Dropout(0.4))

model_2.add(Dense(1, activation='sigmoid'))

In [ ]:
model_2.summary()

In [ ]:
model_2.compile(optimizer="Adam", loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3
)

checkpoint_2 = ModelCheckpoint(
    "models/best_model_2.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

In [ ]:
history_2 = model_2.fit(X_train, y_train, validation_split=0.2, epochs=30, batch_size=32, callbacks=[early_stopping, reduce_lr, checkpoint_2])

In [ ]:
print(history_2.history['accuracy'])

In [ ]:
print(history_2.history['val_accuracy'])

In [ ]:
model_2.evaluate(
    X_test,
    y_test
)

In [ ]:
best_model_2 = load_model("models/best_model_2.keras")

best_model_2.evaluate(
    X_test,
    y_test
)

In [ ]:
y_pred_2 = np.where(model_2.predict(X_test)>0.5, 1, 0)
print(confusion_matrix(y_test,y_pred_2))
print(classification_report(y_test, y_pred_2))

In [ ]:
plt.plot(history_2.history['loss'])
plt.plot(history_2.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'validation'], loc='upper left')
plt.show()

In [ ]:
plt.plot(history_2.history['accuracy'])
plt.plot(history_2.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'validation'], loc='upper left')
plt.show()

In [ ]:
model_2.save_weights("models/weights_best_2.weights.h5")

# Final Performance Evaluation

The final model is evaluated on a held-out test dataset that was not used during training or validation.

Performance is assessed using multiple metrics including Accuracy, Precision, Recall, F1-Score, and Confusion Matrix analysis. These metrics provide a comprehensive understanding of the model's ability to correctly identify both genuine and AI-generated speech.

The selected model demonstrates strong generalization performance and forms the foundation for deployment within the Streamlit-based Deepfake Voice Detection application.


| Model | Conv Blocks | Test Accuracy | Precision | Recall | F1 Score |
|---------|---------|---------|---------|---------|---------|
| CNN V1 | 32-64-128 | 96.19% | 96% | 96% | 96% |
| CNN V2 | 32-64-128-256 | 96.39% | 96% | 96% | 96% |

# Results and Observations

The proposed CNN-based Deepfake Voice Detection system achieved strong classification performance on the test dataset.

Key observations include:

* High classification accuracy on unseen audio samples.
* Balanced precision and recall across both classes.
* Effective detection of AI-generated speech using Mel Spectrogram features.

These results demonstrate the effectiveness of combining audio signal processing with Deep Learning techniques for deepfake voice detection.


# Streamlit Deployment

To make the model accessible to end users, a Streamlit web application is developed.

Users can upload an audio file through the interface, after which the application automatically performs preprocessing, generates Mel Spectrogram features, runs inference using the trained CNN model, and displays whether the audio is REAL or FAKE along with a confidence score.

This transforms the trained model into an interactive real-world application.
